# Lab 01 - Die kleinste vollständige RAG-Pipeline (Teil I)

Sie bauen das RAG-Prinzip in einem Notebook nach: erst **retrieven**, dann
**generieren**. Der Wissensassistent beantwortet Fragen zur Hydraulikpumpe
ausschließlich aus dem mitgelieferten Korpus und belegt jede Aussage mit der
Quellen-Kennung. Genau diese Pipeline (ingestion, index, retrieval, prompting,
answering) ziehen die folgenden Labs Schritt für Schritt auseinander.

Die Embeddings laufen immer lokal (sentence-transformers). Der Generator läuft
über das in `.env` gewählte Backend: lokal via Ollama oder über die
OpenAI-API. Die erste Zelle zeigt, welches gerade aktiv ist.

## 1. Setup und Selbstauskunft

Die erste Zelle zeigt, welches Backend und welches Embedding-Modell aktiv sind.
Läuft Ollama nicht, meldet `common.llm` das mit einer klaren Fehlermeldung.

In [4]:
from common import llm
from common.corpus import load_corpus
from common import retrieval as R

print("LLM-Backend:", llm.active_backend())

docs = load_corpus()
print(f"Korpus: {len(docs)} Dokumente")
for d in docs:
    print(f"  [{d.acl:11s}] {d.doc_id:22s} {d.title}")

LLM-Backend: {'backend': 'openai', 'model': 'gpt-4o-mini', 'ollama_host': 'http://localhost:11434'}
Korpus: 10 Dokumente
  [wartung    ] p12-anzugsmomente      Anzugsmomente Verschraubungen P-12
  [all        ] p12-betriebsdruck      Druckdefinitionen und Betriebsgrenzen P-12
  [all        ] p12-datenblatt         Datenblatt Hydraulikpumpe P-12
  [wartung    ] p12-ersatzteile        Ersatzteilliste P-12
  [all        ] p12-hydraulikoel       Hydrauliköl und Filtration P-12
  [wartung    ] p12-inbetriebnahme     Inbetriebnahme und Entlüftung P-12
  [vertraulich] p12-service-garantie   Service, Garantie und Konditionen P-12 (intern)
  [all        ] p12-sicherheit         Sicherheitshinweise Wartung P-12
  [all        ] p12-stoerungstabelle   Störungstabelle und Fehlercodes P-12
  [all        ] p12-wartungsintervalle Wartungsintervalle und Wartungsplan P-12


## 2. Ingestion und Index

Wir zerlegen die Dokumente in absatzgroße Chunks und bauen einen Hybrid-Index
(BM25 + Dense). Warum Hybrid statt nur Dense? Exakte Begriffe wie Fehlercodes
oder Teile-Nummern findet die lexikalische Suche zuverlässiger; Teil VI geht
dem nach. Für die Pipeline genügt: ein Index, eine `search`-Methode.

In [ ]:
chunks = R.chunk_corpus(docs)

dense_index = R.DenseRetriever(chunks)

In [ ]:
import numpy as np
import pandas as pd

# Die eigentlichen Vektoren stecken in der Embedding-Matrix des Dense-Index:
# eine Zeile pro Chunk, 384 Dimensionen, L2-normiert (Cosine == Skalarprodukt).
vektoren = dense_index._mat
print("Form:", vektoren.shape, "| dtype:", vektoren.dtype)
print("L2-Norm je Vektor (erste 5):", np.linalg.norm(vektoren[:5], axis=1).round(3))

# Vorschau: die ersten 8 Dimensionen je Chunk, beschriftet mit der chunk_id.
pd.DataFrame(
    vektoren[:, :8].round(3),
    index=[c.chunk_id for c in dense_index.chunks],
    columns=[f"dim{i}" for i in range(8)],
)

### Die Vektoren visualisieren

Drei Blickwinkel auf dieselbe Matrix: die Rohaktivierungen (Heatmap), die
paarweise Ähnlichkeit der Chunks und eine 2D-Projektion, die zeigt, wie die
Chunks im Embedding-Raum zueinander liegen.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

ids = [c.chunk_id for c in dense_index.chunks]
doc_ids = [c.doc_id for c in dense_index.chunks]

# 1. Heatmap der Embedding-Matrix: eine Zeile pro Chunk, 384 Spalten (Dimensionen).
#    Jede Zeile ist der Fingerabdruck eines Chunks, kein Muster mit bloßem Auge lesbar.
plt.figure(figsize=(11, 6))
sns.heatmap(vektoren, cmap="vlag", center=0, yticklabels=ids, xticklabels=False,
            cbar_kws={"label": "Aktivierung"})
plt.xlabel("384 Embedding-Dimensionen")
plt.ylabel("Chunk")
plt.title("Embedding-Matrix: eine Zeile pro Chunk")
plt.tight_layout()
plt.show()

In [ ]:
# 2. Ähnlichkeitsmatrix: vektoren @ vektoren.T. Weil die Vektoren L2-normiert sind,
#    ist jeder Eintrag die Cosine-Ähnlichkeit zweier Chunks (Diagonale = 1).
#    Hellere Blöcke entlang der Diagonale: Chunks aus demselben Dokument.
sim = vektoren @ vektoren.T
plt.figure(figsize=(9, 8))
sns.heatmap(sim, cmap="rocket", vmin=0, vmax=1, square=True,
            xticklabels=ids, yticklabels=ids, cbar_kws={"label": "Cosine-Ähnlichkeit"})
plt.title("Chunk-zu-Chunk-Ähnlichkeit (vektoren @ vektoren.T)")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.decomposition import PCA

# 3. 2D-Projektion per PCA: 384 Dimensionen auf 2 heruntergerechnet, nur zur
#    Anschauung. Nahe Punkte sind semantisch ähnlich, Farbe = Herkunftsdokument.
#    Die Ziffer am Punkt ist der Chunk-Index innerhalb des Dokuments.
coords = PCA(n_components=2, random_state=0).fit_transform(vektoren)
df_pca = pd.DataFrame({"x": coords[:, 0], "y": coords[:, 1],
                       "doc_id": doc_ids, "chunk_id": ids})
plt.figure(figsize=(10, 7))
sns.scatterplot(data=df_pca, x="x", y="y", hue="doc_id", s=90)
for _, row in df_pca.iterrows():
    plt.annotate(row["chunk_id"].split("#")[-1], (row["x"], row["y"]),
                 fontsize=7, xytext=(3, 3), textcoords="offset points")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8, title="doc_id")
plt.title("Chunk-Vektoren, 2D-PCA-Projektion")
plt.tight_layout()
plt.show()

### Dense vs Sparse: wann trifft was?

Zwei Retriever, dieselbe Frage. **BM25 (sparse)** matcht exakte Zeichenketten:
Teile-Nummern, Norm- und Typbezeichnungen. **Dense** matcht Bedeutung: es findet
die Stelle auch, wenn die Frage andere Wörter benutzt als der Text. Genau
deshalb kombiniert der Hybrid-Index gleich beide. Ein paar Fragen zeigen den
Unterschied (✓ = erwartetes Dokument steht auf Platz 1):

In [ ]:
sparse_index = R.BM25Retriever(chunks)   # dense_index existiert schon aus Abschnitt 2

# (Frage, erwartetes Dokument, welcher Fall demonstriert wird)
faelle = [
    ("P12-2007",                          "p12-ersatzteile",      "exakter Code -> BM25"),
    ("Anschlussflansch SAE J744",         "p12-datenblatt",       "exakte Bezeichnung -> BM25"),
    ("Wie schwer ist die Pumpe?",         "p12-datenblatt",       "Umschreibung von 'Gewicht' -> Dense"),
    ("Welches Öl soll rein?",             "p12-hydraulikoel",     "umgangssprachlich -> Dense"),
    ("Was bedeutet der Fehlercode E02?",  "p12-stoerungstabelle", "beide treffen"),
]

def top1_doc(retriever, frage):
    (chunk, _), = retriever.search(frage, k=1)
    return chunk.doc_id

def mark(doc, erwartet):
    return ("✓ " if doc == erwartet else "✗ ") + doc

rows = []
for frage, erwartet, fall in faelle:
    rows.append({
        "Frage": frage,
        "BM25 (sparse)": mark(top1_doc(sparse_index, frage), erwartet),
        "Dense": mark(top1_doc(dense_index, frage), erwartet),
        "Fall": fall,
    })
pd.DataFrame(rows)

### Beides fusionieren: der Hybrid-Index (RRF)

Der `HybridRetriever` ist genau das: BM25 und Dense laufen getrennt, dann
verschmilzt **Reciprocal Rank Fusion** ihre beiden Ranglisten. Rangbasiert, per
`score(d) = Summe über beide Listen von 1 / (k + Rang(d))`, mit `k = 60`. Die
rohen Scores von BM25 und Cosine liegen auf verschiedenen Skalen und spielen
deshalb keine Rolle, nur die Platzierung zählt.

Am Code-Fall `P12-2007` sieht man den Effekt: Dense allein lag mit `datenblatt`
daneben (Tabelle oben), aber weil BM25 das richtige `ersatzteile` auf Platz 1
führt und Dense es immerhin auf Platz 2 hat, hebt RRF es an die Spitze. Der
falsche Solo-Treffer wird überstimmt.

In [ ]:
# Der Hybrid-Index, hier explizit gebaut: dieselbe Klasse, die unten als `index`
# den Rest des Labs bedient. Hier machen wir nur die RRF-Fusion sichtbar.
hybrid_index = R.HybridRetriever(chunks, rrf_k=60, pool=20)

frage = "P12-2007"   # der BM25-Fall aus der Vergleichstabelle oben
bm_ranking = [c.chunk_id for c, _ in hybrid_index.bm25.search(frage, 10)]
dn_ranking = [c.chunk_id for c, _ in hybrid_index.dense.search(frage, 10)]
fused = R.reciprocal_rank_fusion([bm_ranking, dn_ranking], rrf_k=hybrid_index.rrf_k)


def rang(ranking, cid):
    return str(ranking.index(cid) + 1) if cid in ranking else "–"


# Fusioniertes Ranking: Rang jeder chunk_id in beiden Listen plus ihr RRF-Score.
pd.DataFrame(
    [{"chunk_id": cid, "Rang BM25": rang(bm_ranking, cid),
      "Rang Dense": rang(dn_ranking, cid), "RRF-Score": round(score, 4)}
     for cid, score in sorted(fused.items(), key=lambda x: -x[1])[:5]]
)

### Feinschliff: LLM-as-Reranker

Retrieval wirft ein weites, billiges Netz (BM25 + Dense). Ein **Reranker** liest
danach die wenigen Top-Kandidaten wirklich und ordnet sie neu. Statt eines
eigenen Cross-Encoders nehmen wir hier das LLM selbst: es bekommt die Frage und
die nummerierten Passagen und gibt die Reihenfolge zurück (RankGPT-Prinzip,
listwise).

Kosten: ein zusätzlicher LLM-Aufruf pro Frage. Nutzen: das Modell liest die
Passagen, wo die Rang-Fusion nur zählt. Beim Beispiel `Wie schwer ist die
Pumpe?` setzt der Hybrid `betriebsdruck` an die Spitze; der Reranker holt das
`datenblatt` mit dem Gewicht nach vorn.

In [ ]:
import re


def llm_rerank(frage, scored_chunks, k=6, max_tokens=200):
    """LLM-as-Reranker (listwise): das LLM ordnet die Kandidaten nach Relevanz neu."""
    kandidaten = [c for c, _ in scored_chunks]
    # Nummerierte Passagen, bewusst gekürzt, damit der Prompt klein bleibt.
    passagen = "\n\n".join(
        f"[{i}] {c.title}: {c.text[:300]}" for i, c in enumerate(kandidaten, 1)
    )
    system = (
        "Du bist ein Reranker. Ordne die Passagen nach Relevanz zur Frage, "
        "die relevanteste zuerst. Antworte NUR mit den Nummern in Reihenfolge, "
        "kommagetrennt, z. B. 3,1,4,2. Keine Erklärung."
    )
    antwort = llm.complete(f"Frage: {frage}\n\nPassagen:\n{passagen}",
                           system=system, temperature=0.0, max_tokens=max_tokens)
    # Robust parsen: Nummern in Reihenfolge, dedupliziert; fehlende hinten anhängen,
    # damit ein unsauber antwortendes Modell nie Kandidaten verliert.
    reihenfolge = []
    for tok in re.findall(r"\d+", antwort):
        i = int(tok) - 1
        if 0 <= i < len(kandidaten) and i not in reihenfolge:
            reihenfolge.append(i)
    reihenfolge += [i for i in range(len(kandidaten)) if i not in reihenfolge]
    return [kandidaten[i] for i in reihenfolge[:k]]


frage = "Wie schwer ist die Pumpe?"
vorher = hybrid_index.search(frage, k=6)
nachher = llm_rerank(frage, vorher)

# Vorher/Nachher im direkten Vergleich: der Reranker zieht das Datenblatt nach oben.
pd.DataFrame({
    "Rang": range(1, len(vorher) + 1),
    "Hybrid": [c.chunk_id for c, _ in vorher],
    "nach LLM-Rerank": [c.chunk_id for c in nachher],
})

In [ ]:



index = R.HybridRetriever(chunks)
print(f"{len(chunks)} Chunks indexiert.")

## 3. Retrieval und Kontext

Für eine Frage holen wir die besten Chunks und formatieren sie zu einem
nummerierten Kontextblock. Jede Quelle trägt ihre Kennung `[doc_id]`, damit
das Modell darauf zitieren kann.

In [ ]:
frage = "Wie hoch ist der zulässige Dauerbetriebsdruck der Pumpe?"
treffer = index.search(frage, k=3)
for c, score in treffer:
    print(f"  {score:.3f}  [{c.doc_id}] {c.title}")

kontext = R.format_context(treffer)
print("\n--- Kontextblock ---\n")
print(kontext[:400], "...")

## 4. Prompting und Antwort

Der System-Prompt legt die Regeln fest: nur aus dem Kontext antworten, mit
`[doc_id]` belegen, sonst "Weiß nicht" sagen. Das ist die kleinste Pipeline,
die belegt oder schweigt. Teil III vertieft das Prompting, Teil IX misst, wie
gut das Grounding tatsächlich hält.

In [ ]:
GROUNDED_SYSTEM = (
    "Sie sind ein technischer Wissensassistent für die Hydraulikpumpe. "
    "Beantworten Sie die Frage ausschließlich anhand des bereitgestellten "
    "Kontexts. Belegen Sie jede Aussage mit der Quellenkennung in eckigen "
    "Klammern, z. B. [p12-betriebsdruck]. Steht die Antwort nicht im Kontext, "
    "antworten Sie genau: 'Das geht aus den Unterlagen nicht hervor.' "
    "Erfinden Sie keine Werte."
)


def baue_prompt(frage: str, kontext: str) -> str:
    return f"Kontext:\n{kontext}\n\nFrage: {frage}\n\nAntwort:"


antwort = llm.complete(baue_prompt(frage, kontext), system=GROUNDED_SYSTEM,
                       temperature=0.0, max_tokens=512)
print(antwort)

## 5. Die Pipeline in einer Funktion

Retrieval, Kontextaufbau und Generierung zusammengefasst. Das ist der
Baustein, den alle weiteren Labs wiederverwenden: `ask(frage)`.

In [ ]:
def ask(frage: str, k: int = 3, max_tokens: int = 300) -> dict:
    treffer = index.search(frage, k=k)
    kontext = R.format_context(treffer)
    antwort = llm.complete(baue_prompt(frage, kontext), system=GROUNDED_SYSTEM,
                           temperature=0.0, max_tokens=max_tokens)
    return {"antwort": antwort, "quellen": [c.doc_id for c, _ in treffer]}


r = ask("Welches Hydrauliköl soll verwendet werden?")
print(r["antwort"])
print("\nHerangezogene Quellen:", r["quellen"])

## Aufgabe 1: Eigene Fragen

Stellen Sie drei eigene Fragen an den Assistenten, von denen mindestens eine
durch den Korpus beantwortbar ist und eine nicht. Prüfen Sie, ob die belegten
Quellen plausibel sind und ob die unbeantwortbare Frage wirklich die
"Weiß nicht"-Antwort auslöst.

In [ ]:
# Ihre Lösung:
meine_fragen = [
    # "...",
]
for f in meine_fragen:
    print("Q:", f)
    print(ask(f)["antwort"], "\n")

### Lösung (eine mögliche)

In [ ]:
for f in [
    "Mit welchem Anzugsmoment werden die Pumpenkopf-Schrauben angezogen?",  # beantwortbar
    "Was bedeutet der Fehlercode E02?",                                      # beantwortbar
    "Wie hoch ist der Stromverbrauch der Steuerelektronik?",                 # nicht im Korpus
]:
    r = ask(f)
    print("Q:", f)
    print("A:", r["antwort"])
    print("Quellen:", r["quellen"], "\n")

## Aufgabe 2: Wert des Retrievals sichtbar machen

Fragen Sie dasselbe Modell einmal **mit** und einmal **ohne** Kontext
(nur die nackte Frage, kein Korpus). Vergleichen Sie die Antworten. Ohne
Grounding rät das Modell oder nennt eine plausibel klingende, aber nicht
belegte Zahl. Das ist das Fehlerbild, gegen das der ganze Kurs arbeitet.

In [ ]:
# Ihre Lösung:
# Tipp: llm.complete(frage, system=...) ohne Kontextblock aufrufen.

### Lösung

In [ ]:
frage = "Auf welchen Wert ist das Druckbegrenzungsventil maximal einzustellen?"
ohne = llm.complete(frage, system="Sie sind ein Hydraulik-Experte. Antworten Sie knapp.",
                    temperature=0.0, max_tokens=512)
mit = ask(frage)
print("OHNE Kontext:\n", ohne)
print("\nMIT Kontext (RAG):\n", mit["antwort"])
print("Quellen:", mit["quellen"])
# Erwartung: nur die RAG-Antwort nennt belegbar 360 bar [p12-betriebsdruck].

## Aufgabe 3: Wie viele Chunks?

Variieren Sie `k` (z. B. 1, 3, 6). Mehr Kontext heißt nicht automatisch
bessere Antworten: irrelevante Chunks verdünnen den Prompt (Teil II,
Lost in the Middle) und kosten Tokens (Teil II, Kosten). Beobachten Sie, ab
wann zusätzlicher Kontext nichts mehr beiträgt.

In [ ]:
# Ihre Lösung:

### Lösung

In [ ]:
frage = "Wann muss das erste Öl gewechselt werden?"
for k in (1, 3, 6):
    r = ask(frage, k=k)
    print(f"k={k}  Quellen={r['quellen']}")
    print("  ", r["antwort"].replace("\n", " ")[:160], "\n")

## Diskussion

- Welche der fünf Bausteine (ingestion, index, retrieval, prompting,
  answering) hat Ihre Antwortqualität hier am stärksten bestimmt?
- Die "Weiß nicht"-Antwort ist eine Funktion, kein Mangel. Wo im eigenen
  Use Case ist ein ehrliches "Weiß nicht" mehr wert als eine Vermutung?
- Diese Pipeline misst noch nichts. Teil VII (Retrieval-Metriken) und Teil IX
  (Grounding) liefern die Zahlen, mit denen Sie sie verbessern.